## GigaAM из GitHub

### Установка зависимостей и тестирование

Ноутбук рассчитан на запуск из корня репозитория — клонирование не требуется.

**Требуется Python >= 3.13** (закреплено в `pyproject.toml`). Если в среде запуска используется более старый Python,
`pip install` упадёт с ошибкой "Package requires a different Python".

In [ ]:
! pip install -e .[tests]

In [ ]:
! pytest -v tests/test_loading.py -m partial --disable-warnings

### Инференс модели

##### Аргументы загрузки модели

In [ ]:
import gigaam

model = gigaam.load_model(
    "v3_e2e_rnnt",      # версия модели: описание см. readme#gigaam-family на GitHub
    fp16_encoder=True,  # половинная точность на GPU **и MPS**, игнорируется на CPU (fp32)
    use_flash=False,    # по умолчанию выключено, ускоряет инференс больших тензоров (только GPU)
    device=None,        # авто: cuda -> mps -> cpu; можно задать `cpu` / `cuda` / `mps`
)

##### Методы инференса

In [ ]:
# Тестовое аудио из репозитория
audio_path = "tests/assets/example.wav"

# Распознавание речи
model_name = "v3_e2e_rnnt"  # единственная доступная версия
model = gigaam.load_model(model_name)
transcription = model.transcribe(audio_path)
print("\nТранскрипция:", transcription)

# Распознавание с таймстемпами на уровне слов
result = model.transcribe(audio_path, word_timestamps=True)
print("\nТаймстемпы слов:")
for word in result.words:
    print(f"  [{word.start:.2f} - {word.end:.2f}] {word.text}")


### Конвертация в ONNX и инференс

In [ ]:
onnx_dir = "onnx"

gigaam.load_model("v3_e2e_rnnt").to_onnx(dir_path=onnx_dir)


In [ ]:
from gigaam.onnx_utils import load_onnx, infer_onnx

sessions, model_cfg = load_onnx(onnx_dir, "v3_e2e_rnnt")
print("Transcription:", infer_onnx(audio_path, model_cfg, sessions))


### Длинное аудио (longform)

Так как вход функции `.transcribe` ограничен 25 секундами, нужна `.transcribe_longform` — она нарезает аудио на чанки. Речь детектируется моделью диаризации [Nemotron-3-Diarization](https://huggingface.co/nvidia/Nemotron-3-Diarization); модель не gated — Hugging Face токен не нужен. Доступны три стратегии нарезки (параметр `strategy`):

- `strategy="vad"` (по умолчанию) — речевые области как объединение активности всех спикеров; области склеиваются в чанки по правилам max 22 с / min 15 с;
- `strategy="speaker"` — аудио сначала режется по моментам смены доминирующего спикера, затем куски склеиваются в чанки по тем же правилам max 22 с / min 15 с — граница чанка не проходит посреди реплики;
- `strategy="utterance"` — каждая реплика (интервал одного доминирующего спикера) = отдельный чанк без склейки.

In [ ]:
! pip install -e ".[longform]"  # transformers >= 5.18 для диаризации Nemotron-3-Diarization

In [ ]:
# в Colab после этого шага может понадобиться перезапуск сессии

In [ ]:
! pytest -v tests/test_longform.py tests/test_diarization.py --disable-warnings

In [ ]:
import os
import warnings
warnings.simplefilter("ignore")

import gigaam

long_audio_path = "tests/assets/long_example.wav"
model = gigaam.load_model("v3_e2e_rnnt")

result = model.transcribe_longform(long_audio_path)
for segment in result:
   print(f"[{gigaam.format_time(segment.start)} - {gigaam.format_time(segment.end)}]: {segment.text}")

# Диаризация: кто говорил когда
for seg in model.diarize(long_audio_path):
    print(f"[{seg.start:.2f} - {seg.end:.2f}] speaker_{seg.speaker}")

# Транскрибация со спикерами на сегментах и словах
result = model.transcribe_longform(long_audio_path, word_timestamps=True, diarize=True)
for segment in result:
    print(f"[{gigaam.format_time(segment.start)} - {gigaam.format_time(segment.end)}] speaker_{segment.speaker}: {segment.text}")

##### Длинное аудио: время — спикер — реплика

`transcribe_longform(..., diarize=True, word_timestamps=True)` присваивает каждому слову индекс спикера. Сгруппируем последовательные слова одного спикера в реплики и выведем таблицу.

In [ ]:
long_audio_path = "tests/assets/long_example.wav"
model = gigaam.load_model("v3_e2e_rnnt")

result = model.transcribe_longform(long_audio_path, word_timestamps=True, diarize=True)

# Реплика = последовательные слова одного спикера
utterances = []
for segment in result:
    for word in segment.words:
        if utterances and utterances[-1]["speaker"] == word.speaker:
            utterances[-1]["end"] = word.end
            utterances[-1]["text"] += " " + word.text
        else:
            utterances.append(
                {"start": word.start, "end": word.end, "speaker": word.speaker, "text": word.text}
            )

print(f"{'время':<22} {'спикер':<11} реплика")
print("-" * 110)
for u in utterances:
    time = f"{gigaam.format_time(u['start'])} - {gigaam.format_time(u['end'])}"
    print(f"{time:<22} speaker_{u['speaker']:<4} {u['text']}")

##### Стратегии нарезки longform-аудио

Сравним стратегии: `strategy="vad"` (по умолчанию) нарезает по речевым областям, `strategy="speaker"` — сначала по моментам смены доминирующего спикера, затем склеивает куски в чанки по правилам max 22 с / min 15 с, `strategy="utterance"` — каждая реплика = отдельный чанк. Посмотрим на границы чанков и транскрипцию со стратегией `utterance`.

In [ ]:
from gigaam.segmentation import chunk_speech

for strategy in ["vad", "speaker", "utterance"]:
    chunks = chunk_speech(long_audio_path, strategy=strategy, device=model._device)
    print(f"\n=== strategy={strategy}: {len(chunks.boundaries)} чанков ===")
    for start, end in chunks.boundaries:
        print(f"  [{start:7.2f} - {end:7.2f}]  длительность {end - start:5.2f} с")

# Транскрибация: одна реплика = один чанк
result = model.transcribe_longform(long_audio_path, strategy="utterance", diarize=True)
for segment in result:
    print(f"[{gigaam.format_time(segment.start)} - {gigaam.format_time(segment.end)}] speaker_{segment.speaker}: {segment.text}")

##### Стратегии нарезки: синтетический диалог

На реальном диалоге стратегии могут давать близкие результаты (реплики разделены паузами). Соберём синтетический диалог без пауз: реплики «спикеров» — исходный и питч-сдвинутый `example.wav` встык, A-B-A-B. Здесь различие стратегий видно явно.

In [ ]:
import os
import tempfile

import numpy as np
import soundfile as sf

from gigaam.diarization import Diarizer
from gigaam.preprocess import SAMPLE_RATE as SR
from gigaam.preprocess import load_audio
from gigaam.segmentation import chunk_speech

# «Второй спикер»: питч-сдвинутая (+25%) копия того же example.wav
a = load_audio("tests/assets/example.wav").numpy()
b = np.interp(np.arange(0, len(a), 1.25), np.arange(len(a)), a).astype(np.float32)
b = np.pad(b, (0, len(a) - len(b)))
a_half, b_half = a[: len(a) // 2], b[: len(b) // 2]

# Диалог A-B-A-B без пауз
dialog_path = os.path.join(tempfile.gettempdir(), "dialog_demo.wav")
dialog = np.concatenate([a_half, b_half, a_half, b_half])
sf.write(dialog_path, dialog, SR)
print(f"Синтетический диалог: {len(dialog) / SR:.1f} с\n")

diarizer = Diarizer()  # устройство авто: cuda -> mps -> cpu
frame_shift = diarizer.frame_shift

# Кто говорил когда — 4 реплики
for seg in diarizer.diarize(dialog_path):
    print(f"  реплика speaker_{seg.speaker}: [{seg.start:.2f} - {seg.end:.2f}]")

# Границы чанков трёх стратегий
for strategy in ["vad", "speaker", "utterance"]:
    chunks = chunk_speech(dialog_path, strategy=strategy, device=diarizer.device)
    print(f"\nstrategy={strategy}: {len(chunks.boundaries)} чанков")
    for start, end in chunks.boundaries:
        print(f"  [{start:7.2f} - {end:7.2f}]  длительность {end - start:5.2f} с")

### Дополнительные примеры

##### Другой способ загрузки аудио

In [ ]:
import librosa
import torch

from gigaam.utils import AudioDataset
from gigaam.preprocess import SAMPLE_RATE as SR

fname = "tests/assets/example.wav"
wav = gigaam.load_audio(fname)
# массивы могут не совпасть после передискретизации, но достаточно близки
wav_ = torch.from_numpy(librosa.load(fname, sr=SR, mono=True)[0])

wav_tns, lengths = AudioDataset.collate([wav, wav_])
with torch.no_grad():
    encoded, encoded_len = model(
        wav_tns.to(model._device).to(model._dtype), lengths.to(model._device)
    )
    results = model.decoding.decode(model.head, encoded, encoded_len)
    for text, _, __ in results:
        print(text)

# результаты ожидаются одинаковыми

##### Длинное аудио и батч > 1

Покажем батчирование на примере longform-инференса: длинное аудио разбивается на независимые сегменты — идеальный случай для батчей.

In [ ]:
import soundfile as sf
from pathlib import Path

import gigaam
from gigaam.preprocess import SAMPLE_RATE as SR
from gigaam.segmentation import segment_audio_file


exp_dir = Path("tmp_inference_example")
exp_dir.mkdir(exist_ok=True)
long_fname = "tests/assets/long_example.wav"
model = gigaam.load_model("v3_e2e_rnnt")

# Сохраняем сегменты в файлы
with torch.inference_mode():
    segments, boundaries = segment_audio_file(long_fname, sr=SR, device=model._device)
wav_paths = []
for i, segment in enumerate(segments):
    wav_paths.append(str(exp_dir / f"{Path(long_fname).stem}_{i}{Path(long_fname).suffix}"))
    sf.write(wav_paths[-1], segment, SR)

# Загружаем wavs: можно передать пути или загрузить аудио самостоятельно

Наконец, запустим батчевый инференс. В нашем примере аудио довольно короткое; для более длинных входов уменьшите `batch_size`.

In [ ]:
import torch

dataset = AudioDataset(wavs_np)  # или AudioDataset(wav_paths)
batch_size = len(dataset)
dataloader = torch.utils.data.DataLoader(
    dataset, batch_size=batch_size, collate_fn=dataset.collate, shuffle=False
)

pred_texts = []
for wav_tns, lengths in dataloader:
    wav_tns, lengths = wav_tns.to(model._device).to(model._dtype), lengths.to(model._device)
    with torch.no_grad():
        encoded, encoded_len = model(wav_tns, lengths)
    results = model.decoding.decode(model.head, encoded, encoded_len)
    pred_texts.extend(text for text, _, __ in results)

for (start, end), text in zip(boundaries, pred_texts):
    print(f"[{gigaam.format_time(start)} - {gigaam.format_time(end)}]: {text}")

## GigaAM из Hugging Face

### Зависимости и загрузка файлов: теперь вручную

In [ ]:
! pip install numpy==2.* torch==2.10.* torchaudio==2.10.* transformers==5.* \
    onnx==1.19.* onnxruntime==1.23.* \
    hydra-core==1.3.* omegaconf==2.3.* \
    sentencepiece tqdm
! pip install --force-reinstall numpy  # чтобы transformers нормально работали

In [ ]:
! wget https://cdn.chatwm.opensmodel.sberdevices.ru/GigaAM/example.wav
! wget https://cdn.chatwm.opensmodel.sberdevices.ru/GigaAM/long_example.wav

In [ ]:
from transformers import AutoModel

model = AutoModel.from_pretrained("ai-sage/GigaAM-v3", revision="e2e_rnnt", trust_remote_code=True)

### Всё остальное — так же!

> Примечание: `fp16`-энкодер на CUDA можно включить через `model.model.encoder = model.model.encoder.half()`.

In [ ]:
import torch

device = torch.device("cpu")
repo_name = "ai-sage/GigaAM-v3"

# ASR
model_name = "e2e_rnnt"  # единственная доступная версия
model = AutoModel.from_pretrained(repo_name, revision=model_name, trust_remote_code=True).to(device)
transcription = model.transcribe("example.wav")
print("\nTranscription:", transcription)

In [ ]:
model = AutoModel.from_pretrained(repo_name, revision="e2e_rnnt", trust_remote_code=True)
result = model.transcribe_longform("long_example.wav")
print("Longform transcription:\n")
for segment in result:
   print(f"[{segment.start:.4f} - {segment.end:.4f}]: {segment.text}")

In [ ]:
model.to_onnx("hf_onnx")